In [ ]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Modelle definition

models_fast = {
    "Naive Baseline (Mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    # n_jobs=-1 nutzt alle CPU-Kerne für RF und KNN
    "Random Forest (Simple)": RandomForestRegressor(n_estimators=50, max_depth=8, n_jobs=-1, random_state=42)
}

models_slow = {
    "KNN Regressor": KNeighborsRegressor(n_neighbors=10, n_jobs=-1) 
}

# 1. Funktion universal

def run_baselines(models_dict, X_train, X_test, y_train, y_test):
    all_results = []
    
    # y_test 
    y_test_original = np.exp(y_test)
    
    for name, model in models_dict.items():
        print(f"⌛ Berechne {name}...")
        start = time.time()
        
        model.fit(X_train, y_train)
        y_pred_log = model.predict(X_test) # this are the log values
        
        duration = time.time() - start

        # calculation original values from the log values
        y_pred_original = np.exp(y_pred_log) 
        
        all_results.append({
            "Model": name,
            "MAE": round(mean_absolute_error(y_test_original, y_pred_original), 2),
            "RMSE": round(np.sqrt(mean_squared_error(y_test_original, y_pred_original)), 2),
            "R2 Score": round(r2_score(y_test_original, y_pred_original), 4),
            "Dauer (sec)": round(duration, 2)
        })
    return pd.DataFrame(all_results)

# 2. run models
results_fast = run_baselines(models_fast, X_train, X_test, y_train, y_test)
results_slow = run_baselines(models_slow, X_train, X_test, y_train, y_test)

# 3. Output final results
final_results = pd.concat([results_fast, results_slow]).sort_values("MAE")
print(final_results)